# PS 1 — Student Performance (UCI)

**DWM practical | Run cells top-to-bottom.**

All datasets are read from `../data/` and results saved in `../outputs/`. Run `python download_datasets.py` in the repo root beforehand.

**Goal:** inspect data, correct dtypes, clean categorical values, one-hot encode features, and normalize numeric features. Student grade `G3` is kept unscaled as the outcome.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler
DATA = Path('../data')
OUT = Path('../outputs'); OUT.mkdir(exist_ok=True)

df = pd.read_csv(DATA/'student-mat.csv', sep=';')
before = df.copy()

In [ ]:
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('Dtypes:'); print(df.dtypes)
print('Missing values:'); print(df.isna().sum())
print('Duplicates:', df.duplicated().sum())
display(df.head())
display(df.describe(include='all').T)

**Cleaning:** UCI represents binary and nominal attributes as strings. Strip whitespace, standardize case, impute missing strings with the mode, and numeric values with the median.

In [ ]:
cat_cols = df.select_dtypes(include=['object','string','category']).columns.tolist()
num_cols = df.select_dtypes(include='number').columns.tolist()
print('Categorical:',cat_cols)
print('Numerical:',num_cols)
for c in cat_cols:
    df[c] = df[c].astype('string').str.strip().str.lower().replace('', pd.NA)
    df[c] = df[c].fillna(df[c].mode(dropna=True).iloc[0]).astype('category')
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')
    df[c] = df[c].fillna(df[c].median())
print(df.dtypes)

**Encoding:** one-hot encoding avoids imposing an arbitrary ranking on nominal categories. **Normalization:** Min-Max puts selected numeric predictors into [0, 1]. Ordinal survey scores are retained as original categories/ratings and final grade `G3` is kept unchanged.

In [ ]:
scale_cols = ['age','absences','failures','G1','G2']
print('Before normalization:'); display(df[scale_cols].head())
df[scale_cols] = MinMaxScaler().fit_transform(df[scale_cols])
print('After normalization:'); display(df[scale_cols].head())
df = pd.get_dummies(df, columns=cat_cols, dtype=int)

In [ ]:
print('BEFORE preprocessing:'); display(before.head())
print('AFTER preprocessing:'); display(df.head())
print('Final shape:', df.shape)
print('Remaining missing values:', df.isna().sum().sum())
print('Remaining duplicates:', df.duplicated().sum())

In [ ]:
df.to_csv(OUT/'PS01_student_processed.csv', index=False)
print('Saved PS01_student_processed.csv')